# 02 · 단일 모델 훈련

여러 모델을 비교하는 벤치마크 대신 **단일 모델만 빠르게 훈련**하고 싶을 때 사용합니다.  
`train_background.py` 엔진(= `mad.benchmark`)을 래핑한 스크립트를 호출합니다.

## ① 파라미터

In [ ]:
from pathlib import Path

# ── 수정 가능한 파라미터 ────────────────────────────────────────────
REPO_DIR       = Path(globals().get('REPO_DIR',       '/content/Military_Object_Detection'))
WORKSPACE_ROOT = Path(globals().get('WORKSPACE_ROOT', '/content/drive/MyDrive/Military_Object_Detection'))
DATASET_YAML   = Path(globals().get('DATASET_YAML',
                       str(WORKSPACE_ROOT / 'data' / 'processed' / 'yolo_dataset' / 'dataset.yaml')))
MODEL_WEIGHTS  = globals().get('MODEL_WEIGHTS', 'yolov8n.pt')  # yolov8n/s/m, yolo11n 등
MODEL_ID       = globals().get('MODEL_ID', 'baseline_single')
OUTPUT_DIR     = Path(globals().get('OUTPUT_DIR',
                       str(WORKSPACE_ROOT / 'experiments' / 'single_model')))
EPOCHS  = int(globals().get('EPOCHS',  20))
IMGSZ   = int(globals().get('IMGSZ',  640))
BATCH   = int(globals().get('BATCH',    8))
WORKERS = int(globals().get('WORKERS',  2))
SEED    = int(globals().get('SEED',    42))
DEVICE  = globals().get('DEVICE', 'auto')
# ──────────────────────────────────────────────────────────────────

print(f'MODEL_WEIGHTS : {MODEL_WEIGHTS}')
print(f'DATASET_YAML  : {DATASET_YAML}')
print(f'OUTPUT_DIR    : {OUTPUT_DIR}')
print(f'EPOCHS / IMGSZ / BATCH : {EPOCHS} / {IMGSZ} / {BATCH}')

## ② 환경 초기화 및 GPU 확인

In [ ]:
import sys, os
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(REPO_DIR)
os.environ['MAD_WORKSPACE_ROOT'] = str(WORKSPACE_ROOT)

from mad.colab_utils import setup_colab_env, check_gpu, check_dataset

setup_colab_env(REPO_DIR, WORKSPACE_ROOT)
check_gpu(require=False)
check_dataset(DATASET_YAML)

## ③ 훈련 실행

In [ ]:
import subprocess, sys

command = [
    sys.executable, 'train_background.py',
    '--workspace-root', str(WORKSPACE_ROOT),
    '--dataset-yaml', str(DATASET_YAML),
    '--model', MODEL_WEIGHTS,
    '--model-id', MODEL_ID,
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--workers', str(WORKERS),
    '--seed', str(SEED),
    '--device', DEVICE,
]

print('실행 명령어:')
print(' '.join(command))
print()
subprocess.run(command, check=True)

## ④ 훈련 결과 확인

In [ ]:
from pathlib import Path
import pandas as pd

exp_dir = OUTPUT_DIR
# 가장 최근 study 디렉토리 찾기
csvs = sorted(exp_dir.rglob('benchmark_all_runs.csv'))
if csvs:
    df = pd.read_csv(csvs[-1])
    print(f'결과 파일: {csvs[-1]}')
    display(df[['model_id', 'seed', 'checkpoint_type', 'val_map50', 'val_map50_95', 'test_map50', 'test_map50_95', 'status']].to_string(index=False))
else:
    print(f'결과를 찾을 수 없습니다: {exp_dir}')

## CLI 동등 명령어
```bash
python train_background.py \\
  --dataset-yaml $MAD_WORKSPACE_ROOT/data/processed/yolo_dataset/dataset.yaml \\
  --model yolov8n.pt --model-id baseline_single \\
  --epochs 20 --imgsz 640 --batch 8 --seed 42
```